In [1]:
"""
Bladder Tissue Classification — Simple Training (No LOPO)
=========================================================
Based on v6 architecture, adapted for simple train/val/test split.
Uses: DINOv2 ViT-B/14 + DenseNet121 (frozen) → CLAM classifier
Target: ≥95% accuracy, ≥90% HGC recall

Usage: python bladder_classification_simple_train.py
"""

'\nBladder Tissue Classification — Simple Training (No LOPO)\n=========================================================\nBased on v6 architecture, adapted for simple train/val/test split.\nUses: DINOv2 ViT-B/14 + DenseNet121 (frozen) → CLAM classifier\nTarget: ≥95% accuracy, ≥90% HGC recall\n\nUsage: python bladder_classification_simple_train.py\n'

# IMPORTS

In [2]:
import os
import sys
import copy
import math
import json
import time
import random
import hashlib
import warnings
from pathlib import Path
from collections import Counter, defaultdict
from datetime import datetime

import numpy as np
import pandas as pd
import cv2
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.models as models

from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score, classification_report,
    confusion_matrix, recall_score, precision_score, f1_score
)

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings('ignore')

# SECTION 1: CONFIGURATION

In [3]:
# --- Path Configurations (Auto-detecting Environment) ---
IS_KAGGLE = os.path.exists('/kaggle')

if IS_KAGGLE:
    # Kaggle environment path resolution
    DATASET_ROOT = '/kaggle/input/ebt-22'  # Fallback default slug for 'EBT 22'
    if os.path.exists('/kaggle/input'):
        try:
            found = False
            # Pass 1: Prioritize 'ebt-22' or similar user dataset
            for root, dirs, files in os.walk('/kaggle/input'):
                dirs[:] = [d for d in dirs if not d.startswith('.')]
                for d in dirs:
                    if 'ebt-22' in d.lower() or 'ebt22' in d.lower():
                        DATASET_ROOT = os.path.join(root, d)
                        found = True
                        break
                if found:
                    break
            
            # Pass 2: Fallback to general 'EndoscopicBladderTissue'
            if not found:
                for root, dirs, files in os.walk('/kaggle/input'):
                    dirs[:] = [d for d in dirs if not d.startswith('.')]
                    for d in dirs:
                        if d == 'EndoscopicBladderTissue':
                            DATASET_ROOT = root  # Parent of EndoscopicBladderTissue
                            found = True
                            break
                    if found:
                        break
        except Exception as e:
            print(f"Warning: Error detecting dataset root in /kaggle/input: {e}")

    ORIG_DATA_DIR = os.path.join(DATASET_ROOT, 'EndoscopicBladderTissue')
    AUG_DATA_BASE = os.path.join(DATASET_ROOT, 'augmented_data_22')
    ANNOTATIONS_CSV = os.path.join(ORIG_DATA_DIR, 'annotations_fixed.csv')
    COMBINED_MANIFEST = os.path.join(AUG_DATA_BASE, 'combined_manifest.csv')
    AUG_MANIFEST = os.path.join(AUG_DATA_BASE, 'augmented_only_manifest.csv')
    OUTPUT_DIR = '/kaggle/working/output'
    CACHE_DIR = '/kaggle/working/feat_cache'
else:
    # Local environment path resolution (workspace relative)
    script_dir = Path(__file__).resolve().parent
    workspace_root = script_dir.parents[1]
    data_dir = workspace_root / 'Data'

    ORIG_DATA_DIR = str(data_dir / 'EndoscopicBladderTissue')
    AUG_DATA_BASE = str(data_dir / 'augmented_data_22')
    ANNOTATIONS_CSV = str(data_dir / 'EndoscopicBladderTissue' / 'annotations_fixed.csv')
    COMBINED_MANIFEST = str(data_dir / 'augmented_data_22' / 'combined_manifest.csv')
    AUG_MANIFEST = str(data_dir / 'augmented_data_22' / 'augmented_only_manifest.csv')
    OUTPUT_DIR = str(workspace_root / 'Output')
    CACHE_DIR = str(workspace_root / 'feat_cache')

print(f"Running environment: {'Kaggle' if IS_KAGGLE else 'Local'}")
print(f"ORIG_DATA_DIR: {ORIG_DATA_DIR}")
print(f"AUG_DATA_BASE: {AUG_DATA_BASE}")
print(f"COMBINED_MANIFEST: {COMBINED_MANIFEST}")

# --- Class Configuration ---
NUM_CLASSES = 3
CLASS_NAMES = ['HGC', 'LGC', 'Normal']
LABEL_MAP = {'HGC': 'HGC', 'LGC': 'LGC', 'NST': 'Normal', 'NTL': 'Normal'}
CLASS_TO_IDX = {'HGC': 0, 'LGC': 1, 'Normal': 2}
IDX_TO_CLASS = {0: 'HGC', 1: 'LGC', 2: 'Normal'}
CANCER_CLASSES = {0, 1}  # HGC, LGC

# --- Image Preprocessing ---
IMAGE_RESIZE = 512
PATCH_SCALES = [96, 128, 192]
PATCH_OUTPUT_SIZE = 224
PATCH_STRIDE_FRAC = 0.5
MIN_TISSUE = 0.40
MAX_BRIGHT = 245
MIN_BRIGHT = 15
MIN_SAT = 10
MIN_FOCUS = 8.0
TOP_QUALITY_FRAC = 0.85
MAX_PATCHES_PER_IMAGE = 60
CLAHE_CLIP = 1.5
CLAHE_GRID = (16, 16)

# --- Feature Extraction ---
FEAT_BATCH = 128
PATCH_BATCH_TARGET = 512
CACHE_VERSION = 'v6_simple_v1'

# --- CLAM Architecture ---
MIL_HIDDEN = 512
MIL_DROPOUT = 0.25
CLAM_K_SAMPLE = 10
N_ATT_HEADS = 4
FEAT_NOISE_STD = 0.02
FEAT_DROP_P = 0.1

# --- Training --- (RUN A: tighter epochs, no Stage 2)
LR = 1e-4
WD = 5e-5
STAGE1_EPOCHS = 40            # RUN A: was 50
STAGE2_EPOCHS = 0             # RUN A: was 15 — DISABLED hard mining
WARMUP_EPOCHS = 3
PATIENCE_S1 = 10              # RUN A: was 12
PATIENCE_S2 = 5
HARD_MINE_FRAC = 0.40
HARD_MINE_LR_FACTOR = 0.2
GRAD_CLIP = 1.0

# --- Loss --- (RUN A: less aggressive focal, lower instance weight)
FOCAL_GAMMA = 2.0             # RUN A: was 3.0
LABEL_SMOOTH = 0.05
BAG_LOSS_W = 1.0
INST_LOSS_W = 0.05            # RUN A: was 0.20 — was hurting attention
HIER_LOSS_W = 0.10
ORDINAL_LOSS_W = 0.10

# --- Class Weighting --- (RUN A)
HGC_WEIGHT_BOOST = 1.5        # RUN A: was 3.0

# --- MixUp ---
MIXUP_ALPHA = 0.3
MIXUP_PROB = 0.20
MIXUP_CAP_FRAC = 0.15

# --- Patch Limits ---
MAX_PATCHES_TRAIN = 200
MAX_PATCHES_TEST = 400

# --- Ensemble --- (RUN A: single model baseline)
N_ENSEMBLE = 1                # RUN A: was 3
ENSEMBLE_DROPOUTS = [0.25]    # RUN A: was [0.20, 0.25, 0.30]
ENSEMBLE_SEEDS = [42]         # RUN A: was [42, 123, 456]

# --- Inference --- (RUN A: disable broken/unhelpful tricks)
USE_MC_DROPOUT = False        # RUN A: was True
MC_DROPOUT_PASSES = 2
USE_TTA = False               # RUN A: was True
TTA_ROUNDS = 3

# --- Balanced Sampling ---
BALANCED_SAMPLING = True

# --- Device ---
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
USE_AMP = torch.cuda.is_available()

# ImageNet normalization constants
IMNET_MEAN = torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1)
IMNET_STD  = torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1)


def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = False
    torch.backends.cudnn.benchmark = True

Running environment: Kaggle
ORIG_DATA_DIR: /kaggle/input/datasets/ronakp004/ebt-22/EndoscopicBladderTissue
AUG_DATA_BASE: /kaggle/input/datasets/ronakp004/ebt-22/augmented_data_22
COMBINED_MANIFEST: /kaggle/input/datasets/ronakp004/ebt-22/augmented_data_22/combined_manifest.csv


# SECTION 2: DATA LOADING

In [4]:
def load_data():
    """Load data from combined_manifest.csv with pre-defined splits."""
    print("\n" + "="*60)
    print("LOADING DATA")
    print("="*60)

    # Try combined manifest first, fall back to annotations + augmented manifest
    if os.path.exists(COMBINED_MANIFEST):
        print(f"Loading combined manifest: {COMBINED_MANIFEST}")
        df = pd.read_csv(COMBINED_MANIFEST)
    else:
        print(f"Combined manifest not found, loading separately...")
        df_orig = pd.read_csv(ANNOTATIONS_CSV)
        df_orig['is_augmented'] = False
        df_orig['aug_mode'] = 'none'
        df_orig['patient_id'] = df_orig['HLY'].apply(extract_patient_id)

        if os.path.exists(AUG_MANIFEST):
            df_aug = pd.read_csv(AUG_MANIFEST)
        else:
            df_aug = pd.DataFrame()

        df = pd.concat([df_orig, df_aug], ignore_index=True)

    # Ensure column names are clean
    df.columns = df.columns.str.strip()

    # Map tissue type to 3-class labels
    df['label'] = df['tissue type'].map(LABEL_MAP)
    df['target'] = df['label'].map(CLASS_TO_IDX)

    # Extract patient ID
    if 'patient_id' not in df.columns:
        df['patient_id'] = df['HLY'].apply(extract_patient_id)
    df['patient'] = df['patient_id']

    # Ensure is_augmented is boolean
    if 'is_augmented' in df.columns:
        df['is_augmented'] = df['is_augmented'].astype(str).str.lower().isin(['true', '1', 'yes'])
    else:
        df['is_augmented'] = False

    # Resolve file paths
    df['path'] = df.apply(resolve_path, axis=1)

    # Split data:
    # - Original images: use explicit patient splits
    # - Augmented images: ONLY add to training set
    df_orig = df[~df['is_augmented']].copy()
    df_aug = df[df['is_augmented']].copy()

    train_patients = [0, 1, 4, 5, 6, 7, 8, 11, 13, 16, 18, 21, 22, 24, 25]
    val_patients = [2, 9, 12, 17]
    test_patients = [10, 14, 23]

    train_df = df_orig[df_orig['patient_id'].isin(train_patients)].copy()
    val_df = df_orig[df_orig['patient_id'].isin(val_patients)].copy()
    test_df = df_orig[df_orig['patient_id'].isin(test_patients)].copy()

    # Add ALL augmented images to training only
    if len(df_aug) > 0:
        train_df = pd.concat([train_df, df_aug], ignore_index=True)
        print(f"  Added {len(df_aug)} augmented images to training set")

    # Print split summary
    for name, split_df in [('Train', train_df), ('Val', val_df), ('Test', test_df)]:
        dist = split_df['label'].value_counts().to_dict()
        n_aug = split_df['is_augmented'].sum() if 'is_augmented' in split_df.columns else 0
        print(f"\n  {name}: {len(split_df)} images ({n_aug} augmented)")
        for cls in CLASS_NAMES:
            print(f"    {cls}: {dist.get(cls, 0)}")

    return train_df, val_df, test_df


def extract_patient_id(filename):
    """Extract patient ID from filename."""
    import re
    patterns = [
        r'case_(\d+)',
        r'cys_case_(\d+)',
    ]
    for pat in patterns:
        m = re.search(pat, str(filename))
        if m:
            return int(m.group(1))
    return 0


def resolve_path(row):
    """Resolve file path for Kaggle or Local filesystem, handling filename mismatches (e.g. _NST_ vs _HLT_)."""
    is_aug = str(row.get('is_augmented', False)).lower() in ['true', '1', 'yes']

    def check_exists_or_alt(p):
        if not p:
            return None
        if os.path.exists(p):
            return p
        
        # Check for filename mismatches where '_NST_' in filename is on disk as '_HLT_', or vice versa.
        dirname, filename = os.path.split(p)
        if '_NST_' in filename:
            alt_filename = filename.replace('_NST_', '_HLT_')
            alt_p = os.path.join(dirname, alt_filename)
            if os.path.exists(alt_p):
                return alt_p
        elif '_HLT_' in filename:
            alt_filename = filename.replace('_HLT_', '_NST_')
            alt_p = os.path.join(dirname, alt_filename)
            if os.path.exists(alt_p):
                return alt_p
                
        # Also try direct replacement in the full path
        if '_NST_' in p:
            alt_p = p.replace('_NST_', '_HLT_')
            if os.path.exists(alt_p):
                return alt_p
        elif '_HLT_' in p:
            alt_p = p.replace('_HLT_', '_NST_')
            if os.path.exists(alt_p):
                return alt_p
                
        return None

    if is_aug:
        # 1. Try constructed path using AUG_DATA_BASE first
        full_path = str(row.get('full_path', ''))
        if full_path:
            candidate = os.path.join(AUG_DATA_BASE, full_path)
            res = check_exists_or_alt(candidate)
            if res:
                return res
            
        # 2. Try raw aug_abs_path if it exists
        aug_path = str(row.get('aug_abs_path', ''))
        res = check_exists_or_alt(aug_path)
        if res:
            return res

        # 3. Try to map legacy path /kaggle/working/augmented_data/ or similar -> AUG_DATA_BASE
        if aug_path:
            for marker in ['/augmented_data/', '/augmented_data_22/']:
                if marker in aug_path:
                    rel_part = aug_path.split(marker)[-1]
                    candidate = os.path.join(AUG_DATA_BASE, rel_part)
                    res = check_exists_or_alt(candidate)
                    if res:
                        return res

        # Fallback candidate (even if it doesn't exist yet, it is the correct expected path)
        if full_path:
            return os.path.join(AUG_DATA_BASE, full_path)
        return aug_path

    else:
        # 1. Try constructed path using ORIG_DATA_DIR
        tissue = str(row.get('tissue type', ''))
        filename = str(row.get('HLY', ''))
        if tissue and filename:
            candidate = os.path.join(ORIG_DATA_DIR, tissue, filename)
            res = check_exists_or_alt(candidate)
            if res:
                return res

        # 2. Try raw abs_path if it exists
        abs_path = str(row.get('abs_path', ''))
        res = check_exists_or_alt(abs_path)
        if res:
            return res

        # 3. Try replacing legacy kaggle input path
        if abs_path:
            legacy_prefixes = [
                '/kaggle/input/datasets/aryashah2k/endoscopic-bladder-tissue-classification-dataset/EndoscopicBladderTissue',
                '/kaggle/input/endoscopic-bladder-tissue-classification-dataset/EndoscopicBladderTissue'
            ]
            for legacy in legacy_prefixes:
                if legacy in abs_path:
                    candidate = abs_path.replace(legacy, ORIG_DATA_DIR)
                    res = check_exists_or_alt(candidate)
                    if res:
                        return res
            
            # Simple replacement of datasets/aryashah2k
            cleaned_abs_path = abs_path.replace('/datasets/aryashah2k/endoscopic-bladder-tissue-classification-dataset/', '/')
            cleaned_abs_path = cleaned_abs_path.replace('/datasets/aryashah2k/', '/')
            res = check_exists_or_alt(cleaned_abs_path)
            if res:
                return res

        # Fallback candidate
        if tissue and filename:
            return os.path.join(ORIG_DATA_DIR, tissue, filename)
        return abs_path

# SECTION 3: IMAGE PREPROCESSING

In [5]:
class LabNormalizer:
    """LAB color normalization (Reinhard-like) + CLAHE."""

    def __init__(self):
        self.ref = None

    def fit(self, images_bgr):
        stats = {'L': [], 'a': [], 'b': []}
        for img in images_bgr:
            lab = cv2.cvtColor(img, cv2.COLOR_BGR2LAB).astype(np.float32)
            for i, ch in enumerate(['L', 'a', 'b']):
                stats[ch].append({
                    'm': lab[:, :, i].mean(),
                    's': lab[:, :, i].std() + 1e-6
                })
        self.ref = {
            ch: {
                'm': np.median([s['m'] for s in stats[ch]]),
                's': np.median([s['s'] for s in stats[ch]])
            } for ch in ['L', 'a', 'b']
        }
        return self

    def transform(self, img_bgr):
        lab = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2LAB).astype(np.float32)
        for i, ch in enumerate(['L', 'a', 'b']):
            c = lab[:, :, i]
            sm, ss = c.mean(), c.std() + 1e-6
            lab[:, :, i] = np.clip(
                (c - sm) * (self.ref[ch]['s'] / ss) + self.ref[ch]['m'], 0, 255
            )
        lab = lab.astype(np.uint8)
        clahe = cv2.createCLAHE(clipLimit=CLAHE_CLIP, tileGridSize=CLAHE_GRID)
        lab[:, :, 0] = clahe.apply(lab[:, :, 0])
        return cv2.cvtColor(lab, cv2.COLOR_LAB2BGR)


def load_image(path, norm=None):
    """Load and resize image."""
    img = cv2.imread(path)
    if img is None:
        raise FileNotFoundError(f"Cannot read image: {path}")
    h, w = img.shape[:2]
    s = IMAGE_RESIZE / max(h, w)
    if s != 1:
        img = cv2.resize(img, (int(w * s), int(h * s)), interpolation=cv2.INTER_AREA)
    if norm:
        img = norm.transform(img)
    return img


def compute_quality(patch_bgr):
    """Compute quality score for a patch."""
    hsv = cv2.cvtColor(patch_bgr, cv2.COLOR_BGR2HSV)
    v = hsv[:, :, 2].astype(np.float32)
    s = hsv[:, :, 1].astype(np.float32)
    mask = (v < MAX_BRIGHT) & (v > MIN_BRIGHT) & (s > MIN_SAT)
    tissue_frac = mask.sum() / mask.size
    if tissue_frac < MIN_TISSUE:
        return -1.0
    gray = cv2.cvtColor(patch_bgr, cv2.COLOR_BGR2GRAY)
    focus = cv2.Laplacian(gray, cv2.CV_64F).var()
    if focus < MIN_FOCUS:
        return -1.0
    focus_norm = min(focus / 100.0, 1.0)
    sat_std = s[mask].std() / 50.0 if mask.sum() > 10 else 0
    sat_norm = min(sat_std, 1.0)
    edges = cv2.Canny(gray, 50, 150)
    edge_density = min(edges.sum() / (255.0 * edges.size) * 10, 1.0)
    return 0.3 * tissue_frac + 0.3 * focus_norm + 0.2 * sat_norm + 0.2 * edge_density


def extract_multiscale_patches(image_bgr, max_patches=None):
    """Multi-scale patch extraction with quality filtering."""
    if max_patches is None:
        max_patches = MAX_PATCHES_PER_IMAGE
    H, W = image_bgr.shape[:2]
    candidates = []
    cap = max_patches * 3
    for scale in PATCH_SCALES:
        if scale > min(H, W):
            continue
        stride = max(1, int(scale * PATCH_STRIDE_FRAC))
        for y in range(0, H - scale + 1, stride):
            for x in range(0, W - scale + 1, stride):
                if len(candidates) >= cap:
                    break
                crop = image_bgr[y:y + scale, x:x + scale]
                q = compute_quality(crop)
                if q > 0:
                    resized = cv2.resize(
                        crop, (PATCH_OUTPUT_SIZE, PATCH_OUTPUT_SIZE),
                        interpolation=cv2.INTER_AREA
                    )
                    candidates.append((resized, q, scale))
            if len(candidates) >= cap:
                break
    if not candidates:
        # Fallback: center crop
        min_dim = min(H, W)
        y0, x0 = (H - min_dim) // 2, (W - min_dim) // 2
        crop = image_bgr[y0:y0 + min_dim, x0:x0 + min_dim]
        resized = cv2.resize(crop, (PATCH_OUTPUT_SIZE, PATCH_OUTPUT_SIZE))
        return [resized], [0.5], [min_dim]
    candidates.sort(key=lambda x: x[1], reverse=True)
    n_keep = max(1, int(len(candidates) * TOP_QUALITY_FRAC))
    candidates = candidates[:n_keep][:max_patches]
    return [c[0] for c in candidates], [c[1] for c in candidates], [c[2] for c in candidates]


def fit_normalizer(df):
    """Fit LAB normalizer on training images."""
    print("  Fitting LAB normalizer...")
    samples = []
    sample_paths = df[~df['is_augmented']].sample(min(50, len(df))).path.values
    for fp in sample_paths:
        try:
            img = cv2.imread(fp)
            if img is not None:
                h, w = img.shape[:2]
                s = IMAGE_RESIZE / max(h, w)
                if s != 1:
                    img = cv2.resize(img, (int(w * s), int(h * s)))
                samples.append(img)
        except Exception:
            pass
    if not samples:
        print("  ⚠ Could not fit normalizer, using identity transform")
        return None
    norm = LabNormalizer().fit(samples)
    print(f"  ✓ Normalizer fitted on {len(samples)} images")
    return norm

# SECTION 4: FEATURE EXTRACTION

In [6]:
# Global backbone models (initialized in main)
dino_model = None
dense_model = None
feat_dim = 0


def load_backbones():
    """Load DINOv2 ViT-B/14 + DenseNet121 backbones."""
    global dino_model, dense_model, feat_dim, IMNET_MEAN, IMNET_STD

    print("\n" + "="*60)
    print("LOADING BACKBONES")
    print("="*60)

    # Move normalization constants to device
    IMNET_MEAN = IMNET_MEAN.to(DEVICE)
    IMNET_STD = IMNET_STD.to(DEVICE)

    # DINOv2
    dino_dim = 0
    print("  Loading DINOv2...")
    for model_name, dim in [('dinov2_vitb14', 768), ('dinov2_vits14', 384)]:
        try:
            dino_model = torch.hub.load('facebookresearch/dinov2', model_name)
            dino_model.eval().to(DEVICE)
            for p in dino_model.parameters():
                p.requires_grad = False
            dino_dim = dim
            print(f"  ✓ {model_name} — dim={dim}")
            break
        except Exception as e:
            print(f"  ⚠ {model_name} failed: {e}")

    if dino_model is None:
        try:
            dino_model = torch.hub.load('facebookresearch/dino:main', 'dino_vits16')
            dino_model.eval().to(DEVICE)
            for p in dino_model.parameters():
                p.requires_grad = False
            dino_dim = 384
            print("  ✓ DINO ViT-S/16 fallback — dim=384")
        except Exception as e:
            print(f"  ⚠ All DINO models failed: {e}")

    # DenseNet121
    densenet = models.densenet121(weights=models.DenseNet121_Weights.DEFAULT)
    dense_dim = densenet.classifier.in_features
    densenet.classifier = nn.Identity()
    dense_model = densenet.eval().to(DEVICE)
    for p in dense_model.parameters():
        p.requires_grad = False
    print(f"  ✓ DenseNet121 — dim={dense_dim}")

    feat_dim = (dino_dim if dino_model else 0) + dense_dim
    print(f"\n  Total feature dim: {feat_dim}")
    return feat_dim


def bgr_to_tensor(patch_bgr):
    """Convert BGR patch to RGB tensor."""
    rgb = cv2.cvtColor(patch_bgr, cv2.COLOR_BGR2RGB)
    return torch.from_numpy(rgb).permute(2, 0, 1).float() / 255.0


def _get_cache_key(path):
    """MD5 cache key for feature caching."""
    key_str = f"{path}|{IMAGE_RESIZE}|{PATCH_SCALES}|{MAX_PATCHES_PER_IMAGE}|{CACHE_VERSION}"
    return hashlib.md5(key_str.encode()).hexdigest()


@torch.inference_mode()
def extract_dual_features(tensor_list):
    """Extract concatenated DINOv2 + DenseNet121 features."""
    if not tensor_list:
        return torch.empty((0, feat_dim), dtype=torch.float16)

    all_feats = []
    for i in range(0, len(tensor_list), FEAT_BATCH):
        batch = torch.stack(tensor_list[i:i + FEAT_BATCH]).to(DEVICE, non_blocking=True)
        batch_norm = (batch - IMNET_MEAN) / IMNET_STD

        parts = []
        with torch.amp.autocast(device_type='cuda', enabled=USE_AMP):
            if dino_model is not None:
                dino_out = dino_model(batch_norm)
                if isinstance(dino_out, dict):
                    dino_feats = dino_out.get('x_norm_clstoken', next(iter(dino_out.values())))
                else:
                    dino_feats = dino_out
                if dino_feats.dim() > 2:
                    dino_feats = dino_feats[:, 0, :]
                parts.append(dino_feats.float().cpu())
            parts.append(dense_model(batch_norm).float().cpu())
        all_feats.append(torch.cat(parts, dim=1))

    return torch.cat(all_feats, 0).half()


def extract_features_for_split(df, desc="Extracting", norm=None, use_cache=True):
    """Extract features for all images in a DataFrame."""
    os.makedirs(CACHE_DIR, exist_ok=True)
    results, skipped, cache_hits = [], 0, 0
    pending_tensors, pending_meta = [], []

    def flush_pending():
        nonlocal pending_tensors, pending_meta
        if not pending_tensors:
            return
        feats = extract_dual_features(pending_tensors)
        start = 0
        for meta in pending_meta:
            n_patches = meta['n_patches']
            feat_block = feats[start:start + n_patches]
            start += n_patches
            if meta['cache_path'] is not None:
                try:
                    torch.save({'features': feat_block}, meta['cache_path'])
                except Exception:
                    pass
            results.append({
                'features': feat_block,
                'label': meta['label'],
                'label_name': meta['label_name'],
                'patient': meta['patient'],
                'path': meta['path'],
                'is_augmented': meta['is_augmented'],
                'aug_mode': meta['aug_mode'],
                'n_patches': feat_block.shape[0],
            })
        pending_tensors.clear()
        pending_meta.clear()

    for _, row in tqdm(df.iterrows(), total=len(df), desc=desc):
        cache_path = None
        if use_cache:
            cache_key = _get_cache_key(str(row.path))
            cache_path = os.path.join(CACHE_DIR, f"{cache_key}.pt")
            if os.path.exists(cache_path):
                try:
                    cached = torch.load(cache_path, map_location='cpu', weights_only=False)
                    results.append({
                        'features': cached['features'],
                        'label': int(row.target),
                        'label_name': row.label,
                        'patient': int(row.patient),
                        'path': row.path,
                        'is_augmented': bool(row.is_augmented),
                        'aug_mode': str(row.get('aug_mode', 'none')),
                        'n_patches': cached['features'].shape[0],
                    })
                    cache_hits += 1
                    continue
                except Exception:
                    pass

        try:
            img = load_image(str(row.path), norm)
        except Exception:
            skipped += 1
            continue

        patches, _, _ = extract_multiscale_patches(img)
        if not patches:
            skipped += 1
            continue

        tensors = [bgr_to_tensor(p) for p in patches]
        if len(tensors) > MAX_PATCHES_PER_IMAGE:
            idx = random.sample(range(len(tensors)), MAX_PATCHES_PER_IMAGE)
            tensors = [tensors[i] for i in sorted(idx)]

        pending_tensors.extend(tensors)
        pending_meta.append({
            'label': int(row.target),
            'label_name': row.label,
            'patient': int(row.patient),
            'path': str(row.path),
            'is_augmented': bool(row.is_augmented),
            'aug_mode': str(row.get('aug_mode', 'none')),
            'n_patches': len(tensors),
            'cache_path': cache_path,
        })

        if len(pending_tensors) >= PATCH_BATCH_TARGET:
            flush_pending()

    flush_pending()

    if skipped:
        print(f"  ⚠ Skipped {skipped} images")
    if cache_hits:
        print(f"  ⚡ Cache hits: {cache_hits}/{len(df)}")
    print(f"  ✓ Extracted features for {len(results)} images")
    return results

# SECTION 5: CLAM MODEL

In [7]:
class MultiHeadGatedAttention(nn.Module):
    """Multi-head gated attention mechanism."""

    def __init__(self, hidden, n_heads=4):
        super().__init__()
        self.n_heads = n_heads
        self.head_dim = hidden // 2 // n_heads
        self.att_nets = nn.ModuleList([
            nn.Sequential(nn.Linear(hidden, self.head_dim), nn.Tanh())
            for _ in range(n_heads)
        ])
        self.gate_nets = nn.ModuleList([
            nn.Sequential(nn.Linear(hidden, self.head_dim), nn.Sigmoid())
            for _ in range(n_heads)
        ])
        self.combine = nn.Linear(self.head_dim * n_heads, hidden // 2)

    def forward(self, h):
        heads = [a(h) * g(h) for a, g in zip(self.att_nets, self.gate_nets)]
        return self.combine(torch.cat(heads, dim=-1))


class CLAM(nn.Module):
    """
    Clustering-constrained Attention Multiple Instance Learning classifier.
    Per-class attention branches + per-class bag/instance classifiers.
    """

    def __init__(self, feat_dim_in, hidden=MIL_HIDDEN, n_classes=NUM_CLASSES,
                 dropout=MIL_DROPOUT, k_sample=CLAM_K_SAMPLE, n_heads=N_ATT_HEADS):
        super().__init__()
        self.n_classes = n_classes
        self.k_sample = k_sample
        self.feat_noise = FEAT_NOISE_STD
        self.feat_drop = nn.Dropout(FEAT_DROP_P)

        # Feature adapter with residual connection
        self.adapter = nn.Sequential(
            nn.Linear(feat_dim_in, feat_dim_in),
            nn.LayerNorm(feat_dim_in),
            nn.GELU(),
            nn.Dropout(0.1),
            nn.Linear(feat_dim_in, feat_dim_in),
        )

        # Encoder
        self.fc = nn.Sequential(
            nn.Linear(feat_dim_in, hidden),
            nn.LayerNorm(hidden),
            nn.GELU(),
            nn.Dropout(dropout),
        )

        # Multi-head gated attention
        self.mh_attention = MultiHeadGatedAttention(hidden, n_heads)

        # Learnable temperature per class
        self.att_temp = nn.Parameter(torch.ones(n_classes))

        # Per-class attention branches
        self.att_branches = nn.ModuleList([
            nn.Linear(hidden // 2, 1) for _ in range(n_classes)
        ])

        # Per-class instance classifiers (for CLAM loss)
        self.inst_classifiers = nn.ModuleList([
            nn.Sequential(
                nn.Linear(hidden, 128), nn.GELU(),
                nn.Dropout(0.1), nn.Linear(128, 2)
            ) for _ in range(n_classes)
        ])

        # Per-class bag classifiers
        self.bag_classifiers = nn.ModuleList([
            nn.Sequential(
                nn.Linear(hidden, hidden // 4),
                nn.GELU(),
                nn.Linear(hidden // 4, 1),
            ) for _ in range(n_classes)
        ])

    def _inst_loss(self, scores, h, classifier, k):
        """Instance-level contrastive loss."""
        N = scores.shape[0]
        k = min(k, N // 2, 8)
        if k < 1:
            return torch.tensor(0.0, device=h.device)
        top_idx = torch.topk(scores, k).indices
        bot_idx = torch.topk(scores, k, largest=False).indices
        feats = torch.cat([h[top_idx], h[bot_idx]], dim=0)
        labels = torch.cat([
            torch.ones(k, dtype=torch.long),
            torch.zeros(k, dtype=torch.long)
        ]).to(h.device)
        return F.cross_entropy(classifier(feats), labels)

    def forward(self, x, label=None):
        x = x.float()

        # Feature noise + dropout during training
        if self.training:
            x = x + torch.randn_like(x) * self.feat_noise
            x = self.feat_drop(x)

        # Adapter with residual
        x = x + self.adapter(x)

        # Encode
        h = self.fc(x)  # (N, hidden)

        # Attention
        att = self.mh_attention(h)  # (N, hidden//2)

        logits = []
        total_inst = torch.tensor(0.0, device=x.device)

        for c in range(self.n_classes):
            a_scores = self.att_branches[c](att).squeeze(-1)  # (N,)
            a_scores = a_scores / (self.att_temp[c].abs() + 0.1)
            a_weights = F.softmax(a_scores, dim=0)  # (N,)
            bag = torch.sum(a_weights.unsqueeze(-1) * h, dim=0)  # (hidden,)
            logits.append(self.bag_classifiers[c](bag))  # (1,)

            # Instance loss for the true class
            if self.training and label is not None and label.item() == c:
                total_inst += self._inst_loss(
                    a_scores.detach(), h,
                    self.inst_classifiers[c], self.k_sample
                )

        return {'logits': torch.cat(logits), 'inst_loss': total_inst}

# SECTION 6: LOSS FUNCTIONS

In [8]:
class FocalLoss(nn.Module):
    """Focal loss with label smoothing."""

    def __init__(self, gamma=2.0, weight=None, label_smoothing=0.0):
        super().__init__()
        self.gamma = gamma
        self.weight = weight
        self.label_smoothing = label_smoothing

    def forward(self, logits, targets):
        ce = F.cross_entropy(
            logits, targets, weight=self.weight,
            label_smoothing=self.label_smoothing, reduction='none'
        )
        pt = torch.exp(-ce)
        return (((1 - pt) ** self.gamma) * ce).mean()


def hierarchical_loss(logits, label):
    """Binary cancer-vs-normal loss."""
    cancer_score = logits[0] + logits[1]
    normal_score = logits[2]
    binary_logit = torch.stack([cancer_score, normal_score]).unsqueeze(0)
    binary_target = torch.tensor(
        [0] if label.item() in CANCER_CLASSES else [1],
        dtype=torch.long, device=label.device
    )
    return F.cross_entropy(binary_logit, binary_target)


def ordinal_loss(logits, label):
    """Ordinal severity loss: HGC(0) > LGC(1) > Normal(2)."""
    probs = F.softmax(logits, dim=0)
    severity = torch.arange(NUM_CLASSES, dtype=torch.float32, device=logits.device)
    pred_severity = (probs * severity).sum()
    true_severity = severity[label]
    return (pred_severity - true_severity) ** 2


def compute_loss(output, label, class_weights=None, focal_criterion=None):
    """Combined loss: focal + hierarchical + ordinal + instance."""
    logits = output['logits'].unsqueeze(0)
    target = label.unsqueeze(0)

    if focal_criterion is not None:
        bag_loss = focal_criterion(logits, target)
    else:
        bag_loss = F.cross_entropy(logits, target, weight=class_weights,
                                   label_smoothing=LABEL_SMOOTH)

    hier = hierarchical_loss(output['logits'], label)
    ordi = ordinal_loss(output['logits'], label)

    total = (BAG_LOSS_W * bag_loss +
             HIER_LOSS_W * hier +
             ORDINAL_LOSS_W * ordi +
             INST_LOSS_W * output['inst_loss'])
    return total

# SECTION 7: TRAINING PIPELINE

In [9]:
def compute_class_weights(data_list):
    """Compute inverse-frequency class weights with HGC boost."""
    counts = Counter(d['label'] for d in data_list)
    total_n = sum(counts.values())
    weights = []
    for c in range(NUM_CLASSES):
        w = total_n / (NUM_CLASSES * max(counts.get(c, 0), 1))
        if c == CLASS_TO_IDX['HGC']:
            w *= HGC_WEIGHT_BOOST
        weights.append(w)
    weight_tensor = torch.tensor(weights, dtype=torch.float32).to(DEVICE)
    print(f"\n  Class weights (HGC boost ×{HGC_WEIGHT_BOOST}):")
    for i, cls in enumerate(CLASS_NAMES):
        print(f"    {cls}: count={counts.get(i, 0)} → weight={weights[i]:.3f}")
    return weight_tensor


def class_balanced_sample(data_list):
    """Oversample minority classes to match majority class count."""
    by_class = defaultdict(list)
    for d in data_list:
        by_class[d['label']].append(d)

    max_count = max(len(v) for v in by_class.values())
    balanced = []
    for cls, items in by_class.items():
        if len(items) >= max_count:
            balanced.extend(items)
        else:
            balanced.extend(items)
            extra = max_count - len(items)
            balanced.extend(random.choices(items, k=extra))

    random.shuffle(balanced)
    return balanced


def feature_mixup(data_list):
    """Feature-space MixUp augmentation. Double frequency for HGC."""
    by_class = defaultdict(list)
    for d in data_list:
        by_class[d['label']].append(d)

    mixed = []
    for cls, items in by_class.items():
        if len(items) < 2:
            continue
        # Double mixup for HGC (class 0)
        n_mix = len(items) if cls in [0, 2] else len(items) // 2
        for _ in range(n_mix):
            if random.random() > MIXUP_PROB:
                continue
            i, j = random.sample(range(len(items)), 2)
            lam = np.random.beta(MIXUP_ALPHA, MIXUP_ALPHA)
            f1, f2 = items[i]['features'], items[j]['features']
            min_n = min(f1.shape[0], f2.shape[0])
            idx1 = torch.randperm(f1.shape[0])[:min_n]
            idx2 = torch.randperm(f2.shape[0])[:min_n]
            mf = lam * f1[idx1] + (1 - lam) * f2[idx2]
            mixed.append({
                'features': mf, 'label': cls,
                'label_name': items[i]['label_name'],
                'patient': items[i]['patient'],
                'path': 'mixup', 'is_augmented': True,
                'aug_mode': 'mixup', 'n_patches': mf.shape[0],
            })
    return mixed


def find_hard_examples(model, data_list, class_weights, focal_criterion):
    """Find top hard_mine_frac hardest examples by loss."""
    model.eval()
    losses = []
    with torch.inference_mode():
        for d in data_list:
            feats = d['features'].to(DEVICE)
            lbl = torch.tensor(d['label'], dtype=torch.long).to(DEVICE)
            if feats.shape[0] > MAX_PATCHES_TEST:
                idx = torch.randperm(feats.shape[0])[:MAX_PATCHES_TEST]
                feats = feats[idx]
            with torch.amp.autocast(device_type='cuda', enabled=USE_AMP):
                out = model(feats, label=lbl)
                loss = compute_loss(out, lbl, class_weights, focal_criterion)
            losses.append(loss.item())

    indexed = sorted(enumerate(losses), key=lambda x: x[1], reverse=True)
    n_hard = max(1, int(len(indexed) * HARD_MINE_FRAC))
    hard_indices = [idx for idx, _ in indexed[:n_hard]]
    return [data_list[i] for i in hard_indices]


def get_warmup_cosine_scheduler(optimizer, warmup_epochs, total_epochs):
    """Warmup-cosine learning rate scheduler."""
    def lr_lambda(epoch):
        if epoch < warmup_epochs:
            return (epoch + 1) / warmup_epochs
        progress = (epoch - warmup_epochs) / max(total_epochs - warmup_epochs, 1)
        return 0.5 * (1.0 + math.cos(math.pi * progress))
    return torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)


def train_clam_two_stage(model, train_images, val_images=None,
                         class_weights=None, dropout_override=None,
                         verbose=True):
    """
    Two-stage training:
    Stage 1: Normal training with balanced sampling + mixup.
    Stage 2: Hard example mining.
    """
    model.train()

    if dropout_override is not None:
        for module in model.modules():
            if isinstance(module, nn.Dropout):
                module.p = dropout_override

    focal_criterion = FocalLoss(
        gamma=FOCAL_GAMMA, weight=class_weights,
        label_smoothing=LABEL_SMOOTH
    ).to(DEVICE)

    optimizer = torch.optim.AdamW(
        model.parameters(), lr=LR, weight_decay=WD
    )
    scheduler = get_warmup_cosine_scheduler(optimizer, WARMUP_EPOCHS, STAGE1_EPOCHS)
    scaler = torch.amp.GradScaler('cuda', enabled=USE_AMP)

    best_val_loss = float('inf')
    best_state = None
    patience_count = 0

    # Generate mixup samples once
    mixup_samples = feature_mixup(train_images)
    mixup_cap = max(1, int(len(train_images) * MIXUP_CAP_FRAC))

    # ── STAGE 1: Main training ───────────────────────────────
    if verbose:
        print(f"\n    Stage 1: {STAGE1_EPOCHS} epochs | {len(train_images)} train | "
              f"{len(mixup_samples)} mixup pool")

    train_losses = []
    val_losses = []

    for epoch in range(STAGE1_EPOCHS):
        model.train()

        # Class-balanced sampling
        if BALANCED_SAMPLING:
            epoch_data = class_balanced_sample(train_images)
        else:
            epoch_data = train_images.copy()

        # Add capped mixup
        if mixup_samples:
            epoch_data.extend(random.sample(
                mixup_samples, min(len(mixup_samples), mixup_cap)
            ))
        random.shuffle(epoch_data)

        epoch_loss = 0.0
        correct = 0
        total = 0
        for img_data in epoch_data:
            feats = img_data['features'].to(DEVICE)
            lbl = torch.tensor(img_data['label'], dtype=torch.long).to(DEVICE)
            if feats.shape[0] > MAX_PATCHES_TRAIN:
                idx = torch.randperm(feats.shape[0])[:MAX_PATCHES_TRAIN]
                feats = feats[idx]

            with torch.amp.autocast(device_type='cuda', enabled=USE_AMP):
                out = model(feats, label=lbl)
                loss = compute_loss(out, lbl, class_weights, focal_criterion)

            optimizer.zero_grad(set_to_none=True)
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
            scaler.step(optimizer)
            scaler.update()

            epoch_loss += loss.item()
            pred = out['logits'].argmax().item()
            correct += int(pred == lbl.item())
            total += 1

        scheduler.step()
        avg_train_loss = epoch_loss / max(total, 1)
        train_acc = correct / max(total, 1)
        train_losses.append(avg_train_loss)

        # Validation
        if val_images:
            model.eval()
            val_loss = 0.0
            val_correct = 0
            with torch.inference_mode():
                for img_data in val_images:
                    feats = img_data['features'].to(DEVICE)
                    lbl = torch.tensor(img_data['label'], dtype=torch.long).to(DEVICE)
                    if feats.shape[0] > MAX_PATCHES_TEST:
                        idx = torch.randperm(feats.shape[0])[:MAX_PATCHES_TEST]
                        feats = feats[idx]
                    with torch.amp.autocast(device_type='cuda', enabled=USE_AMP):
                        out = model(feats, label=lbl)
                        val_loss += compute_loss(out, lbl, class_weights, focal_criterion).item()
                    pred = out['logits'].argmax().item()
                    val_correct += int(pred == lbl.item())

            avg_val = val_loss / max(len(val_images), 1)
            val_acc = val_correct / max(len(val_images), 1)
            val_losses.append(avg_val)

            if avg_val < best_val_loss:
                best_val_loss = avg_val
                best_state = copy.deepcopy(model.state_dict())
                patience_count = 0
            else:
                patience_count += 1

            if patience_count >= PATIENCE_S1:
                if verbose:
                    print(f"    ✓ Stage 1 early stop at epoch {epoch + 1}")
                break

            if verbose and ((epoch + 1) % 5 == 0 or epoch == 0):
                print(f"    Epoch {epoch + 1:3d}/{STAGE1_EPOCHS} | "
                      f"train_loss={avg_train_loss:.4f} train_acc={train_acc:.3f} | "
                      f"val_loss={avg_val:.4f} val_acc={val_acc:.3f} | "
                      f"patience={patience_count}/{PATIENCE_S1}")

    # Load best stage 1 model
    if best_state is not None:
        model.load_state_dict(best_state)

    # ── STAGE 2: Hard example mining ─────────────────────────
    if STAGE2_EPOCHS > 0:
        if verbose:
            print(f"\n    Stage 2: Hard mining — {STAGE2_EPOCHS} epochs")

        hard_examples = find_hard_examples(
            model, train_images, class_weights, focal_criterion
        )
        if verbose:
            hard_dist = Counter(d['label'] for d in hard_examples)
            print(f"    Hard examples: {len(hard_examples)} — "
                  f"{', '.join(f'{IDX_TO_CLASS[k]}:{v}' for k, v in sorted(hard_dist.items()))}")

        # Lower learning rate
        for pg in optimizer.param_groups:
            pg['lr'] = LR * HARD_MINE_LR_FACTOR

        best_val_loss_s2 = float('inf')
        patience_count_s2 = 0

        for epoch in range(STAGE2_EPOCHS):
            model.train()
            random.shuffle(hard_examples)
            epoch_loss = 0.0

            for img_data in hard_examples:
                feats = img_data['features'].to(DEVICE)
                lbl = torch.tensor(img_data['label'], dtype=torch.long).to(DEVICE)
                if feats.shape[0] > MAX_PATCHES_TRAIN:
                    idx = torch.randperm(feats.shape[0])[:MAX_PATCHES_TRAIN]
                    feats = feats[idx]

                with torch.amp.autocast(device_type='cuda', enabled=USE_AMP):
                    out = model(feats, label=lbl)
                    loss = compute_loss(out, lbl, class_weights, focal_criterion)

                optimizer.zero_grad(set_to_none=True)
                scaler.scale(loss).backward()
                scaler.unscale_(optimizer)
                nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
                scaler.step(optimizer)
                scaler.update()
                epoch_loss += loss.item()

            if val_images:
                model.eval()
                val_loss = 0.0
                with torch.inference_mode():
                    for img_data in val_images:
                        feats = img_data['features'].to(DEVICE)
                        lbl = torch.tensor(img_data['label'], dtype=torch.long).to(DEVICE)
                        if feats.shape[0] > MAX_PATCHES_TEST:
                            idx = torch.randperm(feats.shape[0])[:MAX_PATCHES_TEST]
                            feats = feats[idx]
                        with torch.amp.autocast(device_type='cuda', enabled=USE_AMP):
                            out = model(feats, label=lbl)
                            val_loss += compute_loss(out, lbl, class_weights, focal_criterion).item()

                avg_val = val_loss / max(len(val_images), 1)
                if avg_val < best_val_loss_s2:
                    best_val_loss_s2 = avg_val
                    best_state = copy.deepcopy(model.state_dict())
                    patience_count_s2 = 0
                else:
                    patience_count_s2 += 1

                if patience_count_s2 >= PATIENCE_S2:
                    if verbose:
                        print(f"    ✓ Stage 2 early stop at epoch {epoch + 1}")
                    break

                if verbose and ((epoch + 1) % 5 == 0 or epoch == 0):
                    print(f"    S2 Epoch {epoch + 1:3d}/{STAGE2_EPOCHS} | "
                          f"loss={epoch_loss / len(hard_examples):.4f} | "
                          f"val={avg_val:.4f}")

        if best_state is not None:
            model.load_state_dict(best_state)

    return model, train_losses, val_losses

# SECTION 8: EVALUATION

In [10]:
@torch.no_grad()
def predict_image_single(model, feats, use_mc=False):
    """Single model prediction, optionally with MC Dropout."""
    if use_mc:
        # Only enable Dropout layers, not other training-specific things
        model.eval()
        for m in model.modules():
            if isinstance(m, nn.Dropout):
                m.train()
    else:
        model.eval()

    feats = feats.to(DEVICE)
    if feats.shape[0] > MAX_PATCHES_TEST:
        idx = torch.randperm(feats.shape[0])[:MAX_PATCHES_TEST]
        feats = feats[idx]

    with torch.amp.autocast(device_type='cuda', enabled=USE_AMP):
        out = model(feats)
    probs = F.softmax(out['logits'].float(), dim=0)
    return probs.cpu().numpy()


def predict_image_ensemble(models_list, feats, use_tta=False):
    """
    Ensemble prediction with:
    1. Standard prediction
    2. MC Dropout passes
    3. TTA (random patch subsets)
    4. Confidence-weighted geometric mean
    """
    all_probs = []
    all_entropy = []

    for model in models_list:
        model_probs = []

        # Standard prediction
        p = predict_image_single(model, feats, use_mc=False)
        model_probs.append(p)

        # MC Dropout passes
        if USE_MC_DROPOUT:
            for _ in range(MC_DROPOUT_PASSES):
                p_mc = predict_image_single(model, feats, use_mc=True)
                model_probs.append(p_mc)

        # TTA: random patch subsets
        if use_tta and USE_TTA:
            for _ in range(TTA_ROUNDS):
                p_tta = predict_image_single(model, feats, use_mc=False)
                model_probs.append(p_tta)

        # Average this model's predictions
        avg_p = np.mean(model_probs, axis=0)
        avg_p = avg_p / (avg_p.sum() + 1e-8)

        # Entropy as confidence measure
        entropy = -np.sum(avg_p * np.log(avg_p + 1e-8))
        all_probs.append(avg_p)
        all_entropy.append(entropy)

    # Confidence-weighted geometric mean
    all_probs = np.array(all_probs)
    all_entropy = np.array(all_entropy)

    weights = 1.0 / (all_entropy + 0.1)
    weights = weights / weights.sum()

    log_probs = np.log(all_probs + 1e-8)
    weighted_log = np.sum(log_probs * weights[:, None], axis=0)
    geo_mean = np.exp(weighted_log)
    geo_mean = geo_mean / geo_mean.sum()

    return np.argmax(geo_mean), geo_mean.max(), geo_mean


def evaluate_ensemble(models, data, desc="Evaluating"):
    """Evaluate ensemble on a dataset."""
    all_preds = []
    all_labels = []
    all_probs = []

    for sample in tqdm(data, desc=desc):
        feats = sample['features']
        label = sample['label']

        pred, conf, probs = predict_image_ensemble(models, feats, use_tta=True)
        all_preds.append(pred)
        all_labels.append(label)
        all_probs.append(probs)

    return all_preds, all_labels, all_probs


def print_metrics(preds, labels, title="Results"):
    """Print comprehensive evaluation metrics."""
    print(f"\n{'='*60}")
    print(f"  {title}")
    print(f"{'='*60}")

    acc = accuracy_score(labels, preds)
    bal_acc = balanced_accuracy_score(labels, preds)

    print(f"\n  Accuracy:          {acc:.4f} ({acc*100:.1f}%)")
    print(f"  Balanced Accuracy: {bal_acc:.4f} ({bal_acc*100:.1f}%)")

    print(f"\n  Classification Report:")
    report = classification_report(labels, preds, target_names=CLASS_NAMES, digits=4)
    for line in report.split('\n'):
        print(f"    {line}")

    # HGC-specific metrics
    hgc_idx = CLASS_TO_IDX['HGC']
    hgc_mask_true = [l == hgc_idx for l in labels]
    hgc_mask_pred = [p == hgc_idx for p in preds]

    hgc_true_count = sum(hgc_mask_true)
    hgc_correct = sum(1 for t, p in zip(labels, preds) if t == hgc_idx and p == hgc_idx)
    hgc_recall = hgc_correct / max(hgc_true_count, 1)
    hgc_precision = hgc_correct / max(sum(hgc_mask_pred), 1)

    print(f"\n  HGC Metrics:")
    print(f"    Recall:    {hgc_recall:.4f} ({hgc_recall*100:.1f}%)")
    print(f"    Precision: {hgc_precision:.4f} ({hgc_precision*100:.1f}%)")

    # Confusion matrix
    cm = confusion_matrix(labels, preds)
    print(f"\n  Confusion Matrix:")
    print(f"    {'':>10s}", end='')
    for cls in CLASS_NAMES:
        print(f"  {cls:>8s}", end='')
    print()
    for i, cls in enumerate(CLASS_NAMES):
        print(f"    {cls:>10s}", end='')
        for j in range(NUM_CLASSES):
            print(f"  {cm[i, j]:>8d}", end='')
        print()

    # Target check
    print(f"\n  ╔══════════════════════════════════════════╗")
    acc_pass = acc >= 0.95
    hgc_pass = hgc_recall >= 0.90
    print(f"  ║  Accuracy ≥ 95%:    {'✅ PASS' if acc_pass else '❌ FAIL'} ({acc*100:.1f}%)  ║")
    print(f"  ║  HGC Recall ≥ 90%:  {'✅ PASS' if hgc_pass else '❌ FAIL'} ({hgc_recall*100:.1f}%)  ║")
    overall = acc_pass and hgc_pass
    print(f"  ║  Overall:            {'✅ ALL TARGETS MET' if overall else '❌ TARGETS NOT MET'}    ║")
    print(f"  ╚══════════════════════════════════════════╝")

    return {
        'accuracy': acc,
        'balanced_accuracy': bal_acc,
        'hgc_recall': hgc_recall,
        'hgc_precision': hgc_precision,
        'confusion_matrix': cm.tolist(),
        'targets_met': overall,
    }


def save_confusion_matrix(labels, preds, output_path):
    """Save confusion matrix as figure."""
    cm = confusion_matrix(labels, preds)
    fig, ax = plt.subplots(figsize=(8, 6))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES, ax=ax)
    ax.set_xlabel('Predicted')
    ax.set_ylabel('True')
    ax.set_title('Confusion Matrix — Test Set')
    plt.tight_layout()
    plt.savefig(output_path, dpi=150)
    plt.close()
    print(f"  Saved confusion matrix to {output_path}")

# SECTION 9: MAIN

In [11]:
def main():
    """Main training and evaluation pipeline."""
    start_time = time.time()

    print("╔══════════════════════════════════════════════════════════╗")
    print("║  Bladder Tissue Classification — Simple Training        ║")
    print("║  Architecture: DINOv2 + DenseNet121 → CLAM              ║")
    print("║  Target: ≥95% Accuracy, ≥90% HGC Recall                 ║")
    print("╚══════════════════════════════════════════════════════════╝")
    print(f"\nDevice: {DEVICE}")
    print(f"AMP: {USE_AMP}")

    os.makedirs(OUTPUT_DIR, exist_ok=True)
    os.makedirs(CACHE_DIR, exist_ok=True)

    # ── 1. Load data ─────────────────────────────────────────
    train_df, val_df, test_df = load_data()

    # ── 2. Load backbones ────────────────────────────────────
    actual_feat_dim = load_backbones()

    # ── 3. Fit normalizer ────────────────────────────────────
    norm = fit_normalizer(train_df)

    # ── 4. Extract features ──────────────────────────────────
    print("\n" + "="*60)
    print("EXTRACTING FEATURES")
    print("="*60)

    train_data = extract_features_for_split(train_df, "Train features", norm)
    val_data = extract_features_for_split(val_df, "Val features", norm)
    test_data = extract_features_for_split(test_df, "Test features", norm)

    print(f"\n  Train: {len(train_data)} | Val: {len(val_data)} | Test: {len(test_data)}")

    # ── 5. Compute class weights ─────────────────────────────
    class_weights = compute_class_weights(train_data)

    # ── 6. Train ensemble ────────────────────────────────────
    print("\n" + "="*60)
    print("TRAINING ENSEMBLE")
    print("="*60)

    ensemble_models = []
    all_train_losses = []
    all_val_losses = []

    for i in range(N_ENSEMBLE):
        seed = ENSEMBLE_SEEDS[i]
        dropout = ENSEMBLE_DROPOUTS[i]
        set_seed(seed)

        print(f"\n{'─'*50}")
        print(f"  Ensemble Member {i + 1}/{N_ENSEMBLE}")
        print(f"  Seed: {seed} | Dropout: {dropout}")
        print(f"{'─'*50}")

        model = CLAM(
            feat_dim_in=actual_feat_dim,
            hidden=MIL_HIDDEN,
            n_classes=NUM_CLASSES,
            dropout=dropout,
            k_sample=CLAM_K_SAMPLE,
            n_heads=N_ATT_HEADS,
        ).to(DEVICE)

        n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
        print(f"  Trainable parameters: {n_params:,}")

        model, t_losses, v_losses = train_clam_two_stage(
            model, train_data, val_data,
            class_weights=class_weights,
            dropout_override=dropout,
            verbose=True,
        )

        ensemble_models.append(model)
        all_train_losses.append(t_losses)
        all_val_losses.append(v_losses)

        # Save individual model checkpoint
        checkpoint_path = os.path.join(OUTPUT_DIR, f'model_ensemble_{i + 1}.pt')
        torch.save({
            'model_state_dict': model.state_dict(),
            'seed': seed,
            'dropout': dropout,
            'feat_dim': actual_feat_dim,
        }, checkpoint_path)
        print(f"  ✓ Saved checkpoint: {checkpoint_path}")

    # ── 7. Evaluate on validation set ────────────────────────
    print("\n" + "="*60)
    print("VALIDATION SET EVALUATION")
    print("="*60)

    val_preds, val_labels, val_probs = evaluate_ensemble(
        ensemble_models, val_data, "Validating"
    )
    val_metrics = print_metrics(val_preds, val_labels, "VALIDATION RESULTS")

    # ── 8. Evaluate on test set ──────────────────────────────
    print("\n" + "="*60)
    print("TEST SET EVALUATION")
    print("="*60)

    test_preds, test_labels, test_probs = evaluate_ensemble(
        ensemble_models, test_data, "Testing"
    )
    test_metrics = print_metrics(test_preds, test_labels, "TEST RESULTS")

    # ── 9. Save outputs ─────────────────────────────────────
    # Confusion matrix figure
    save_confusion_matrix(
        test_labels, test_preds,
        os.path.join(OUTPUT_DIR, 'confusion_matrix_test.png')
    )

    # Training curves
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    for i, (t_loss, v_loss) in enumerate(zip(all_train_losses, all_val_losses)):
        axes[0].plot(t_loss, label=f'Model {i + 1}', alpha=0.7)
        if v_loss:
            axes[1].plot(v_loss, label=f'Model {i + 1}', alpha=0.7)
    axes[0].set_title('Training Loss')
    axes[0].set_xlabel('Epoch')
    axes[0].legend()
    axes[1].set_title('Validation Loss')
    axes[1].set_xlabel('Epoch')
    axes[1].legend()
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, 'training_curves.png'), dpi=150)
    plt.close()

    # Metrics JSON
    elapsed = time.time() - start_time
    results = {
        'timestamp': datetime.now().isoformat(),
        'runtime_minutes': elapsed / 60,
        'config': {
            'num_classes': NUM_CLASSES,
            'class_names': CLASS_NAMES,
            'hgc_boost': HGC_WEIGHT_BOOST,
            'focal_gamma': FOCAL_GAMMA,
            'lr': LR,
            'stage1_epochs': STAGE1_EPOCHS,
            'stage2_epochs': STAGE2_EPOCHS,
            'ensemble_size': N_ENSEMBLE,
            'feat_dim': actual_feat_dim,
        },
        'data': {
            'train_total': len(train_data),
            'val_total': len(val_data),
            'test_total': len(test_data),
        },
        'validation': val_metrics,
        'test': test_metrics,
    }

    with open(os.path.join(OUTPUT_DIR, 'results.json'), 'w') as f:
        json.dump(results, f, indent=2, default=str)

    # ── 10. Final summary ────────────────────────────────────
    print(f"\n{'='*60}")
    print(f"  TRAINING COMPLETE")
    print(f"{'='*60}")
    print(f"  Total runtime: {elapsed / 60:.1f} minutes")
    print(f"  Test accuracy: {test_metrics['accuracy']*100:.1f}%")
    print(f"  HGC recall:    {test_metrics['hgc_recall']*100:.1f}%")
    print(f"  Targets met:   {'✅ YES' if test_metrics['targets_met'] else '❌ NO'}")
    print(f"\n  Outputs saved to: {OUTPUT_DIR}")


if __name__ == '__main__':
    main()

╔══════════════════════════════════════════════════════════╗
║  Bladder Tissue Classification — Simple Training        ║
║  Architecture: DINOv2 + DenseNet121 → CLAM              ║
║  Target: ≥95% Accuracy, ≥90% HGC Recall                 ║
╚══════════════════════════════════════════════════════════╝

Device: cuda
AMP: True

LOADING DATA
Loading combined manifest: /kaggle/input/datasets/ronakp004/ebt-22/augmented_data_22/combined_manifest.csv
  Added 1717 augmented images to training set

  Train: 2961 images (1717 augmented)
    HGC: 804
    LGC: 1116
    Normal: 1041

  Val: 265 images (0 augmented)
    HGC: 77
    LGC: 85
    Normal: 103

  Test: 245 images (0 augmented)
    HGC: 57
    LGC: 93
    Normal: 95

LOADING BACKBONES
  Loading DINOv2...
Downloading: "https://github.com/facebookresearch/dinov2/zipball/main" to /root/.cache/torch/hub/main.zip
Downloading: "https://dl.fbaipublicfiles.com/dinov2/dinov2_vitb14/dinov2_vitb14_pretrain.pth" to /root/.cache/torch/hub/checkpoints/d

100%|██████████| 330M/330M [00:01<00:00, 335MB/s] 


  ✓ dinov2_vitb14 — dim=768
Downloading: "https://download.pytorch.org/models/densenet121-a639ec97.pth" to /root/.cache/torch/hub/checkpoints/densenet121-a639ec97.pth


100%|██████████| 30.8M/30.8M [00:00<00:00, 166MB/s] 


  ✓ DenseNet121 — dim=1024

  Total feature dim: 1792
  Fitting LAB normalizer...
  ✓ Normalizer fitted on 50 images

EXTRACTING FEATURES


...

  ✓ Extracted features for 245 images

  Train: 2961 | Val: 265 | Test: 245

  Class weights (HGC boost ×1.5):
    HGC: count=804 → weight=1.841
    LGC: count=1116 → weight=0.884
    Normal: count=1041 → weight=0.948

TRAINING ENSEMBLE

──────────────────────────────────────────────────
  Ensemble Member 1/1
  Seed: 42 | Dropout: 0.25
──────────────────────────────────────────────────
  Trainable parameters: 8,073,103

    Stage 1: 40 epochs | 2961 train | 485 mixup pool
    Epoch   1/40 | train_loss=0.3497 train_acc=0.831 | val_loss=0.6324 val_acc=0.758 | patience=0/10
    Epoch   5/40 | train_loss=0.1502 train_acc=0.947 | val_loss=0.6826 val_acc=0.872 | patience=1/10
    Epoch  10/40 | train_loss=0.0723 train_acc=0.979 | val_loss=0.4401 val_acc=0.887 | patience=1/10
    Epoch  15/40 | train_loss=0.0318 train_acc=0.991 | val_loss=0.3695 val_acc=0.913 | patience=4/10
    Epoch  20/40 | train_loss=0.0197 train_acc=0.995 | val_loss=0.5055 val_acc=0.917 | patience=9/10
    ✓ Stage 1 earl

Validating:   0%|          | 0/265 [00:00<?, ?it/s]


  VALIDATION RESULTS

  Accuracy:          0.8906 (89.1%)
  Balanced Accuracy: 0.8918 (89.2%)

  Classification Report:
                  precision    recall  f1-score   support
    
             HGC     0.7624    1.0000    0.8652        77
             LGC     0.9275    0.7529    0.8312        85
          Normal     1.0000    0.9223    0.9596       103
    
        accuracy                         0.8906       265
       macro avg     0.8966    0.8918    0.8853       265
    weighted avg     0.9077    0.8906    0.8910       265
    

  HGC Metrics:
    Recall:    1.0000 (100.0%)
    Precision: 0.7624 (76.2%)

  Confusion Matrix:
                     HGC       LGC    Normal
           HGC        77         0         0
           LGC        21        64         0
        Normal         3         5        95

  ╔══════════════════════════════════════════╗
  ║  Accuracy ≥ 95%:    ❌ FAIL (89.1%)  ║
  ║  HGC Recall ≥ 90%:  ✅ PASS (100.0%)  ║
  ║  Overall:            ❌ TARGETS NOT MET    ║

Testing:   0%|          | 0/245 [00:00<?, ?it/s]


  TEST RESULTS

  Accuracy:          0.7878 (78.8%)
  Balanced Accuracy: 0.8111 (81.1%)

  Classification Report:
                  precision    recall  f1-score   support
    
             HGC     0.5392    0.9649    0.6918        57
             LGC     0.9722    0.7527    0.8485        93
          Normal     0.9577    0.7158    0.8193        95
    
        accuracy                         0.7878       245
       macro avg     0.8231    0.8111    0.7865       245
    weighted avg     0.8659    0.7878    0.8007       245
    

  HGC Metrics:
    Recall:    0.9649 (96.5%)
    Precision: 0.5392 (53.9%)

  Confusion Matrix:
                     HGC       LGC    Normal
           HGC        55         1         1
           LGC        21        70         2
        Normal        26         1        68

  ╔══════════════════════════════════════════╗
  ║  Accuracy ≥ 95%:    ❌ FAIL (78.8%)  ║
  ║  HGC Recall ≥ 90%:  ✅ PASS (96.5%)  ║
  ║  Overall:            ❌ TARGETS NOT MET    ║
  ╚════